# 02 · Baseline prompt → image test runs
**Milestone 1 deliverable: 5 sample prompt→image runs**

Uses **Stable Diffusion 1.5** conditioned on the CLIP embeddings cached in notebook 01.
Every run uses a fixed seed and logs its settings so results are reproducible.

> Needs a GPU. On a Colab T4, each 512×512 image takes roughly 5–10 s at 30 steps.

In [ ]:
# --- Setup: works in Google Colab and locally ---
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/words-to-worlds.git"   # <-- change to your repo
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("words-to-worlds").exists():
        !git clone -q {REPO_URL}
    %cd words-to-worlds
    !pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")              # run from repo root locally

sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
import json, torch
import matplotlib.pyplot as plt
import pandas as pd
from src import config as C
from src.data import load_image
from src.generate import load_pipeline, generate, save_run

pipe = load_pipeline(C.SD_MODEL, scheduler=C.SCHEDULER)
print("Loaded on", pipe.device, "| scheduler:", type(pipe.scheduler).__name__)

## 1. Pick 5 COCO captions as prompts

In [ ]:
emb = torch.load(C.PROC_DIR / "clip_text_1k.pt")
subset = pd.read_json(C.PROC_DIR / "pairs_1k.json")

idx = torch.Generator().manual_seed(C.SEED)
pick = torch.randperm(len(emb["captions"]), generator=idx)[:5].tolist()
prompts = [emb["captions"][i] for i in pick]
prompt_embeds = emb["hidden"][pick]           # (5, 77, 768) from our own pipeline
for i, p in enumerate(prompts): print(i, p)

## 2. Generate from our cached CLIP embeddings

In [ ]:
settings = dict(model=C.SD_MODEL, scheduler=C.SCHEDULER, guidance_scale=C.GUIDANCE_SCALE,
                num_steps=C.NUM_STEPS, seed=C.SEED, size=C.IMAGE_SIZE, conditioning="cached CLIP ViT-L/14 hidden states")

images, secs = generate(pipe, prompt_embeds=prompt_embeds, guidance_scale=C.GUIDANCE_SCALE,
                        num_steps=C.NUM_STEPS, seed=C.SEED, size=C.IMAGE_SIZE)
print(f"Generated {len(images)} images in {secs:.1f}s ({secs/len(images):.1f}s each)")
save_run(images, prompts, C.SAMPLES_DIR, settings, prefix="baseline")

In [ ]:
# Generated vs real COCO image
real_files = [subset.loc[subset.image_id == emb["image_ids"][i], "file_name"].item() for i in pick]
IMG_DIR = C.PROC_DIR / f"images_{C.IMAGE_SIZE}"

fig, axes = plt.subplots(2, 5, figsize=(20, 8.5))
for j, (img, p, fn) in enumerate(zip(images, prompts, real_files)):
    axes[0, j].imshow(img); axes[0, j].set_title(p[:55] + ("…" if len(p) > 55 else ""), fontsize=9)
    axes[1, j].imshow(load_image(IMG_DIR / fn))
for ax in axes.flat: ax.axis("off")
axes[0, 0].text(-40, 256, "generated", rotation=90, va="center", fontsize=12)
axes[1, 0].text(-40, 256, "real COCO", rotation=90, va="center", fontsize=12)
plt.tight_layout(); plt.savefig(C.SAMPLES_DIR / "baseline_vs_real.png", dpi=100, bbox_inches="tight"); plt.show()

## 3. Check: cached embeddings = pipeline's own text encoder
Same prompt, same seed, two routes. The images should be (near) identical, which proves our text→embedding pipeline matches what SD expects.

In [ ]:
a, _ = generate(pipe, prompt_embeds=prompt_embeds[:1], seed=C.SEED, num_steps=C.NUM_STEPS)
b, _ = generate(pipe, prompts=prompts[:1], seed=C.SEED, num_steps=C.NUM_STEPS)
import numpy as np
diff = np.abs(np.asarray(a[0], dtype=int) - np.asarray(b[0], dtype=int)).mean()
print(f"Mean pixel difference (0–255): {diff:.2f}  -> small (< ~2) means the routes match")

## 4. Preview for Milestone 2: guidance scale
One prompt, same seed, different classifier-free guidance. Low = loose/diverse, high = literal/saturated.

In [ ]:
scales = [1.0, 3.0, 7.5, 12.0, 20.0]
p = prompts[0]
grid = [generate(pipe, prompts=[p], guidance_scale=g, num_steps=C.NUM_STEPS, seed=C.SEED)[0][0] for g in scales]

fig, axes = plt.subplots(1, len(scales), figsize=(20, 4.5))
for ax, img, g in zip(axes, grid, scales):
    ax.imshow(img); ax.set_title(f"guidance = {g}"); ax.axis("off")
fig.suptitle(p, fontsize=11); plt.tight_layout()
plt.savefig(C.SAMPLES_DIR / "guidance_preview.png", dpi=100, bbox_inches="tight"); plt.show()

## Observations (fill in for the 1-page summary)
- Which prompts worked well / badly? (counting, text in images, small objects, hands are common failures)
- How did guidance change realism vs. prompt-faithfulness?
- Generation time per image on our GPU: ___ s